In [1]:
# 사용 데이터 : 감정 라벨 분류 데이터로 fine-tuning 시키고 task 를 진행시켜보자!
# Fine-tuning 전략 : LoRA & 하이퍼파라미터 튜닝은 Optuna 로 진행
# 제대로 감정이 분류되려면 EDA 시 1. 과연 해당 감정 라벨의 감정은 그 감정인가 확인. 2. 각 감정 데이터의 비율 확인 3. 각 감정 데이터의 특징 간단 확인
# 각 라벨별 데이터 분포 확인 하고 train 시 데이터 균형 맞추기 위해가중치 두기 from sklearn.utils.class_weight import compute_class_weight

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install optuna
!pip install seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 19.8 MB/s eta 0:00:00


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from collections import Counter

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score, roc_auc_score, confusion_matrix
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType

import optuna

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# 데이터 받기

In [5]:
train_df = pd.read_csv('/content/drive/MyDrive/data/training.csv')
test_df = pd.read_csv('/content/drive/MyDrive/data/test.csv')
val_df = pd.read_csv('/content/drive/MyDrive/data/validation.csv') # 동일 폴더 안에 있을 때는 파일명만 써도 됨

In [6]:
train_df.head()

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [7]:
test_df.head()

,text,label
0,im feeling rather rotten so im not very ambiti...,0
1,im updating my blog because i feel shitty,0
2,i never make her separate from me because i do...,0
3,i left with my bouquet of red and yellow tulip...,1
4,i was feeling a little vain when i did this one,0


# 결측치 확인

In [8]:
print(train_df.isnull().sum())
print(test_df.isnull().sum())
print(val_df.isnull().sum())

text     0
label    0
dtype: int64
text     0
label    0
dtype: int64
text     0
label    0
dtype: int64


In [9]:
label_map = {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'} # 라벨 6개 데이터
train_df['label_name'] = train_df['label'].map(label_map) # 라벨 이름 딕셔너리 대로 매핑 label_name 에 추가
val_df['label_name'] = val_df['label'].map(label_map)
test_df['label_name'] = test_df['label'].map(label_map)

# 라벨 분포 확인 : 분포가 train, test, val 끼리 비슷함을 알 수 있음. 다만 불균형 문제는 있다.

학습 시 라벨별로 가중치를 두면 되지 않을까

In [10]:
train_df['label_name'].value_counts()

,count
label_name,
joy,5362
sadness,4666
anger,2159
fear,1937
love,1304
surprise,572


In [11]:
train_df['label_name'].value_counts(normalize=True) * 100


,proportion
label_name,
joy,33.51250
sadness,29.16250
anger,13.49375
fear,12.10625
love,8.15000
surprise,3.57500


In [12]:
test_df['label_name'].value_counts()

,count
label_name,
joy,695
sadness,581
anger,275
fear,224
love,159
surprise,66


In [13]:
test_df['label_name'].value_counts(normalize=True) * 100 # 0-1 정규화 후 곱하기 100

,proportion
label_name,
joy,34.75
sadness,29.05
anger,13.75
fear,11.20
love,7.95
surprise,3.30


In [14]:
val_df['label_name'].value_counts()

,count
label_name,
joy,704
sadness,550
anger,275
fear,212
love,178
surprise,81


In [15]:
val_df['label_name'].value_counts(normalize=True) * 100

,proportion
label_name,
joy,35.20
sadness,27.50
anger,13.75
fear,10.60
love,8.90
surprise,4.05


## 중복 데이터 확인

In [16]:
print(f"Train 중복: {train_df.duplicated(subset='text').sum()} / {len(train_df)}")
print(f"Val   중복: {val_df.duplicated(subset='text').sum()} / {len(val_df)}")
print(f"Test  중복: {test_df.duplicated(subset='text').sum()} / {len(test_df)}")

Train 중복: 31 / 16000
Val   중복: 2 / 2000
Test  중복: 0 / 2000


In [17]:
# 각 그 중복 데이터가 무엇인지 확인
# subset - 중복 판단 기준 컬럼 설정
# keep - 중복 중 어떤 것을 중복이 아닌 것으로 할지
# keep=False 이면 원본 포함 모든 중복 데이터 표시. keep=first 는 중복 첫 값은 표시 안하고 나머지만 표시

train_df[train_df.duplicated(subset='text', keep=False)].sort_values('text')

,text,label,label_name
8246,i am not amazing or great at photography but i...,2,love
3508,i am not amazing or great at photography but i...,1,joy
15705,i began to feel accepted by gaia on her own terms,1,joy
5277,i began to feel accepted by gaia on her own terms,2,love
8804,i bet taylor swift basks in the knowledge that...,4,fear
...,...,...,...
11354,i write these words i feel sweet baby kicks fr...,2,love
7685,im still not sure why reilly feels the need to...,5,surprise
2908,im still not sure why reilly feels the need to...,4,fear
9596,ive also made it with both sugar measurements ...,1,joy


In [18]:
pd.set_option('display.max_colwidth', None) #
train_df.loc[[3508, 8246]] # 특정 칼럼 묶어서 뽑아 표시

,text,label,label_name
3508,i am not amazing or great at photography but i feel passionate about it,1,joy
8246,i am not amazing or great at photography but i feel passionate about it,2,love


In [19]:
train_df.loc[[9596, 9069]]

,text,label,label_name
9596,ive also made it with both sugar measurements but i feel like cup is just too sweet for me,1,joy
9069,ive also made it with both sugar measurements but i feel like cup is just too sweet for me,2,love


In [20]:
val_df[val_df.duplicated(subset='text', keep=False)].sort_values('text') # text 기준으로 뽑고 중복된거 전부 표시

,text,label,label_name
774,i feel so tortured by it,4,fear
1993,i feel so tortured by it,3,anger
300,i have had several new members tell me how comfortable they feel with how accepted they are by the existing members and that is great to hear,2,love
603,i have had several new members tell me how comfortable they feel with how accepted they are by the existing members and that is great to hear,1,joy


## 텍스트 길이 분포 (라벨별) → max_length 설정 근거

In [21]:
train_df['text'].str.len().describe() # max_len = 300

,text
count,16000.000000
mean,96.845812
std,55.904953
min,7.000000
25%,53.000000
50%,86.000000
75%,129.000000
max,300.000000


## 영어가 아닌 정보가 무엇이 있는지 확인

In [22]:
import re
chars = set(re.findall(r'[^a-zA-Z\s]', ' '.join(train_df['text'])))
print(sorted(chars)) #re.findall(패턴, 문자열). 영어가 아니고 공백 아닌거 반환해라.
# 결과를 보니 모든 문자가 영어 -> 전처리 필요 없는 깨끗한 데이터

[]


# 모델 불러오기

In [23]:
model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=6)
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Dataset 만들고 토치용 DataLoader 에 올리기

In [24]:
# BERT 계열 토크나이저의 경우 토큰화 + 벡터화 동시 진행.
# (batch, seq_len, dim) 이 기본 파이토치 텍스트 차원수
# BERT / Transformer 계열은 무조건 위 순서.
# 이미지는 (batch, channels, height, width)
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=300):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts) # 문장 길이를 알아야 토큰화 진행하니까

    def __getitem__(self, index):
        encoding = self.tokenizer( #tokenizer() 에 return_tensors="pt" 넣으면 배치 차원 추가함.
            self.texts[index],
            truncation=True,       # max_len 초과 시 잘라냄
            padding='max_length',  # max_len까지 패딩
            max_length=self.max_len, #지정한 최대 길이
            return_tensors='pt'    # PyTorch 텐서로 반환
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),        # (max_len,)
            'attention_mask': encoding['attention_mask'].squeeze(0), # (max_len,)
            'labels': torch.tensor(self.labels[index], dtype=torch.long)
        }

train_dataset = EmotionDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
val_dataset = EmotionDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer)
test_dataset = EmotionDataset(test_df['text'].tolist(), test_df['label'].tolist(), tokenizer)

In [25]:
sample = train_dataset[0]
print(sample) #Dataset 에 올린건 일단 토큰 id 인 것

{'input_ids': tensor([  101,   177, 34420, 10123, 38008, 26506, 55177, 89771,   102,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

In [26]:
print(f"  input_ids shape: {sample['input_ids'].shape}")
print(f"  attention_mask shape: {sample['attention_mask'].shape}")
print(f"  label: {sample['labels']}")

  input_ids shape: torch.Size([300])
  attention_mask shape: torch.Size([300])
  label: 0


# 데이터로더에 올리기 -> 배치로 묶는 과정

In [27]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

batch = next(iter(train_loader))
print(f"input_ids: {batch['input_ids'].shape}")  #배치, 문장 길이
print(f"attention_mask: {batch['attention_mask'].shape}")
print(f"labels: {batch['labels'].shape}")

input_ids: torch.Size([32, 300])
attention_mask: torch.Size([32, 300])
labels: torch.Size([32])


## 데이터 가중치

In [28]:
# 적은 클래스(surprise 3.5%)에 높은 가중치, 많은 클래스(joy 33%)에 낮은 가중치
# 가중치 = 전체 샘플 수 / (클래스 수 × 해당 클래스 샘플 수)
# sklearn.utils.class_weight import compute_class_weight 으로 반환값이 넘파이라 텐서 변환 필요

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label'].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [29]:
print(class_weights) # 0-5 라벨

tensor([0.5715, 0.4973, 2.0450, 1.2351, 1.3767, 4.6620], device='cuda:0')


# 모델 가져오고 LoRA 적용

In [30]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence Classification
    r=8,                         # 줄이는 차원수.
    lora_alpha=16,               # 스케일링 -
    lora_dropout=0.1,
    target_modules=['query', 'value']  # attention의 Q, V에만 LoRA 적용
)

model = get_peft_model(model, lora_config)
model.to(device)

model.print_trainable_parameters()  # 전체 대비 학습 파라미터 비율 확인

trainable params: 299,526 || all params: 178,157,580 || trainable%: 0.1681


# 학습 루프

In [31]:
EPOCHS = 20 #에폭
LR = 2e-5 #learning rate

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss(weight=class_weights)  # 클래스 가중치 적용

for epoch in range(EPOCHS):
    # train 데이터로 학습
    model.train()
    total_loss = 0
    train_preds, train_labels = [], []

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        train_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        # 아웃풋 텐서 중 마지막 축 기준 (클래스 분류 기준) 가장 높은 로짓값 반환.
        # 매 배치마다 예측결과를 extend 로 쌓기
        train_labels.extend(labels.cpu().numpy())

    train_acc = accuracy_score(train_labels, train_preds)

    # validation 데이터로 eval
    model.eval()
    val_preds, val_labels_list = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            val_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
            val_labels_list.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels_list, val_preds)
    val_f1 = f1_score(val_labels_list, val_preds, average='weighted') # 가중치 둔 f1 스코어

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

Epoch 1/20 | Loss: 1.7995 | Train Acc: 0.1366 | Val Acc: 0.1360 | Val F1: 0.0848
Epoch 2/20 | Loss: 1.7914 | Train Acc: 0.1972 | Val Acc: 0.2755 | Val F1: 0.2336
Epoch 3/20 | Loss: 1.7762 | Train Acc: 0.2532 | Val Acc: 0.3095 | Val F1: 0.3137
Epoch 4/20 | Loss: 1.7015 | Train Acc: 0.2881 | Val Acc: 0.3000 | Val F1: 0.2938
Epoch 5/20 | Loss: 1.5668 | Train Acc: 0.3359 | Val Acc: 0.3865 | Val F1: 0.3818
Epoch 6/20 | Loss: 1.4868 | Train Acc: 0.3657 | Val Acc: 0.4040 | Val F1: 0.4084
Epoch 7/20 | Loss: 1.4171 | Train Acc: 0.4015 | Val Acc: 0.4515 | Val F1: 0.4577
Epoch 8/20 | Loss: 1.3596 | Train Acc: 0.4407 | Val Acc: 0.4795 | Val F1: 0.4892
Epoch 9/20 | Loss: 1.3052 | Train Acc: 0.4744 | Val Acc: 0.5110 | Val F1: 0.5186
Epoch 10/20 | Loss: 1.2407 | Train Acc: 0.5005 | Val Acc: 0.5190 | Val F1: 0.5253
Epoch 11/20 | Loss: 1.1951 | Train Acc: 0.5270 | Val Acc: 0.5480 | Val F1: 0.5548
Epoch 12/20 | Loss: 1.1504 | Train Acc: 0.5399 | Val Acc: 0.5555 | Val F1: 0.5648
Epoch 13/20 | Loss: 1.118

# Test 데이터로 평가

In [32]:
# 평가 모드 전환
model.eval()
test_preds, test_labels_list, test_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        test_preds.extend(outputs.logits.argmax(dim=-1).cpu().numpy())
        test_probs.extend(torch.softmax(outputs.logits, dim=-1).cpu().numpy())
        test_labels_list.extend(labels.cpu().numpy())

test_probs = np.array(test_probs)

print(f"  1. Accuracy:       {accuracy_score(test_labels_list, test_preds):.4f}")
print(f"  2. F1 (weighted):  {f1_score(test_labels_list, test_preds, average='weighted'):.4f}")
print(f"  3. F1 (macro):     {f1_score(test_labels_list, test_preds, average='macro'):.4f}")
print(f"  4. AUC :      {roc_auc_score(test_labels_list, test_probs, multi_class='ovr', average='weighted'):.4f}")

  1. Accuracy:       0.6245
  2. F1 (weighted):  0.6367
  3. F1 (macro):     0.5917
  4. AUC :      0.9152
